# Most connected cities – long-distance services (2025)
Which cities have the most **direct long-distance trains** to other cities, over the full enriched dataset.

- A journey is long-distance if any of its rows has `LD == "Yes"` (the flag can vary along one journey, e.g. EC Wien–Bratislava).
- Stations are grouped into cities with Trainline `stations.csv` (e.g. Wien Hbf + Wien Meidling → Wien).
- **Close capitals** (e.g. Wien–Bratislava): pairs between two `CAPITALS` also count on non-LD trains.
- One train = one date + train number, so a train reported by several agencies is counted once.
- Low RAM: the parquet is streamed in 500k-row batches and strings are hashed to integers (~1.4 GB peak, ~1.5 min).

### 1. Setup

In [7]:
import re, unicodedata, numpy as np, pandas as pd, pyarrow.parquet as pq

DATA = "intermediate_outputs/data_chuuchuu_2025_01_to_2025_12_enriched.parquet"
STATIONS = "sup_data/stations.csv"
MIN_TRAINS = 52   # a city pair counts as "connected" with >= 52 direct trains in the year (~weekly)
# Close capitals linked by regional trains (e.g. Wien-Bratislava): pairs among these also count on non-LD services
CAPITALS = {"Wien", "Bratislava", "Budapest", "Praha", "Berlin", "Bruxelles", "Amsterdam", "Luxembourg", "Bern",
            "Paris", "Roma", "Ljubljana", "Zagreb", "København", "Warszawa", "Vaduz", "Monaco"}
pf = pq.ParquetFile(DATA)
scan = lambda cols: (b.to_pandas() for b in pf.iter_batches(batch_size=500_000, columns=cols))  # streamed, low RAM
h = lambda s: pd.util.hash_array(s.to_numpy())                                                    # string -> uint64
in_sorted = lambda a, ref: ref[np.minimum(np.searchsorted(ref, a), len(ref) - 1)] == a            # fast np.isin

### 2. Station -> city (Trainline stations.csv: climb parents to the "is_city" node, else city-name prefix within 20 km)

In [8]:
st = pd.read_csv(STATIONS, sep=";", dtype=str, usecols=["id", "name", "db_id", "parent_station_id", "is_city", "latitude", "longitude"])
st[["latitude", "longitude"]] = st[["latitude", "longitude"]].astype(float)
norm = lambda x: re.sub(r"[^a-z]", "", unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower())
st["key"] = st.name.map(norm)
s = st.set_index("id")
cities = st[st.is_city.eq("t") & st.latitude.notna()]
by_db, by_key = st.drop_duplicates("db_id").set_index("db_id"), st.drop_duplicates("key").set_index("key")

def to_city(i):
    r = s.loc[i]
    for _ in range(5):
        if s.at[i, "is_city"] == "t": return s.at[i, "name"]
        i = s.at[i, "parent_station_id"]
        if pd.isna(i): break
    if pd.notna(r.latitude):   # e.g. "Amsterdam Bijlmer ArenA" -> Amsterdam
        km = 111 * np.hypot(cities.latitude - r.latitude, (cities.longitude - r.longitude) * np.cos(np.radians(r.latitude)))
        c = cities[km < 20]
        c = c[[r.key.startswith(k) for k in c.key]]
        if len(c): return c.loc[c.key.str.len().idxmax(), "name"]

def resolve(db_id, stop_name):
    k = norm(stop_name)
    r = by_db.loc[db_id] if db_id in by_db.index else None
    if (r is None or r.key[:4] != k[:4]) and k in by_key.index:   # Italian stop ids collide with Trainline ids -> match by name
        r = by_key.loc[k]
    city = to_city(r.id) if r is not None else None
    name = r["name"] if r is not None and pd.notna(r["name"]) else stop_name
    if pd.isna(name): return None
    return city or re.sub(r"[ -](Hbf|Hauptbahnhof|Centraal|Centrale|Central|Bahnhof)$", "", name)   # fallback: station name

# pass 1: unique stops + set of long-distance journeys (journey is LD if any of its rows is LD)
stops, ld_j, days = [], [], set()
for df in scan(["journey_id", "deutscheBahnStopId", "stopName", "country", "LD", "date"]):
    days |= set(df.date.unique())
    stops.append(df[["deutscheBahnStopId", "stopName", "country"]].drop_duplicates("deutscheBahnStopId"))
    ld_j.append(np.unique(h(df.journey_id[df.LD == "Yes"])))
stops = pd.concat(stops).drop_duplicates("deutscheBahnStopId").set_index("deutscheBahnStopId")
stop_country, stops = stops.country, stops.stopName
ld_j, n_days = np.unique(np.concatenate(ld_j)), len(days)
stop_city = pd.Series([resolve(i, n) for i, n in stops.items()], index=stops.index).dropna()
city_names = pd.Index(stop_city.unique())
stop_code = pd.Series(city_names.get_indexer(stop_city), index=stop_city.index, dtype="int32")
city_country = stop_country.groupby(stop_city).agg(lambda s: s.mode().iat[0] if s.notna().any() else None)  # most common country of its stops
cap_codes = city_names.get_indexer(sorted(CAPITALS & set(city_names)))
print(f"{len(stops):,} stops -> {len(city_names):,} cities | {len(ld_j):,} LD journeys | capitals:", sorted(CAPITALS & set(city_names)))

9,025 stops -> 8,317 cities | 2,051,763 LD journeys | capitals: ['Amsterdam', 'Berlin', 'Bern', 'Bratislava', 'Bruxelles', 'Budapest', 'København', 'Ljubljana', 'Luxembourg', 'Paris', 'Praha', 'Roma', 'Warszawa', 'Wien', 'Zagreb']


### 3. Pass 2: (journey, city) for LD journeys + capital stops of other journeys

In [9]:
parts, trains = [], []
for df in scan(["journey_id", "deutscheBahnStopId"]):
    x = pd.DataFrame({"j": h(df.journey_id), "c": stop_code.reindex(df.deutscheBahnStopId, fill_value=-1).to_numpy()})
    keep = (in_sorted(x.j.to_numpy(), ld_j) | x.c.isin(cap_codes)) & (x.c >= 0)
    parts.append(x[keep].drop_duplicates())
    ids = pd.Series(df.journey_id[keep].unique())   # journey_id = <agency>_<type>_<number>_<date>: same train seen by several agencies
    trains.append(pd.DataFrame({"j": h(ids), "t": h(ids.str.extract(r"([^_]*_[^_]*)$")[0])}))
jc = pd.concat(parts, ignore_index=True); del parts
train_of = pd.concat(trains).drop_duplicates("j").set_index("j").t; del trains
jc = jc[in_sorted(jc.j.to_numpy(), ld_j) | (jc.groupby("j").c.transform("size") > 1)]   # non-LD journeys only kept if they link 2 capitals
jc["t"] = train_of.reindex(jc.j).to_numpy(); del train_of
print(f"{len(jc):,} journey-city rows ({jc.memory_usage().sum()/1e6:.0f} MB) over {n_days} days")

17,827,698 journey-city rows (499 MB) over 365 days


### 4. Direct trains per city pair (chunked self-join; a train reported by several agencies is counted once)

In [ ]:
counts = []
for k in range(32):
    x = jc[jc.t % 32 == k]
    p = x.merge(x, on=["j", "t"])
    p = p[p.c_x < p.c_y].drop_duplicates(["t", "c_x", "c_y"])
    counts.append(p.groupby(["c_x", "c_y"]).size())
pairs = pd.concat(counts).groupby(level=[0, 1]).sum().reset_index(name="direct_trains")
pairs = pd.DataFrame({"city_a": city_names[pairs.c_x], "city_b": city_names[pairs.c_y], "direct_trains": pairs.direct_trains,
                      "trains_per_day": (pairs.direct_trains / n_days).round(1)})
pairs["type"] = np.where(pairs.city_a.map(city_country) == pairs.city_b.map(city_country), "domestic", "international")
pairs = pairs.sort_values("direct_trains", ascending=False, ignore_index=True)
display(pairs.head(30))

,city_a,city_b,direct_trains,trains_per_day,type
0,Amsterdam,Schiphol Airport,133808,366.6,domestic
1,Utrecht,Amsterdam,127989,350.7,domestic
2,Rotterdam,Amsterdam,106162,290.9,domestic
3,Den Haag,Leiden,102352,280.4,domestic
4,Amsterdam,Leiden,102293,280.3,domestic
...,...,...,...,...,...
72870,Quaregnon,Couillet,1,0.0,domestic
72871,Quaregnon,Aiseau,1,0.0,domestic
72872,Quaregnon,Jemeppe-sur-Sambre,1,0.0,domestic
72873,Quaregnon,Tubize,1,0.0,domestic


In [14]:
pairs.to_excel("summary_stats/city_connections.xlsx", index=False)

OSError: Cannot save file into a non-existent directory: '\summary_stats'

### 5. Most connected cities: number of distinct cities with >= MIN_TRAINS direct trains

In [11]:
strong = pairs[pairs.direct_trains >= MIN_TRAINS]
both = pd.concat([strong.iloc[:, :3].set_axis(["city", "to", "n"], axis=1), strong[["city_b", "city_a", "direct_trains"]].set_axis(["city", "to", "n"], axis=1)])
ranking = (both.groupby("city").agg(connected_cities=("to", "size"), direct_trains=("n", "sum"))
               .sort_values(["connected_cities", "direct_trains"], ascending=False))
ranking.head(30)

,connected_cities,direct_trains
city,,
Wien,372,1139275
München,323,999018
Berlin,301,973054
Frankfurt (Main),298,1259853
Zürich,289,1760561
Stuttgart,286,711725
Graz,249,378522
Paris,246,586987
Rosenheim,243,308052


### 6. Close-capital pairs (check Wien–Bratislava)

In [12]:
pairs[pairs.city_a.isin(CAPITALS) & pairs.city_b.isin(CAPITALS)].head(20)

,city_a,city_b,direct_trains,trains_per_day,type
483,Bruxelles,Paris,24471,67.0,international
1362,Bruxelles,Amsterdam,15188,41.6,international
1499,Wien,Budapest,13919,38.1,international
2783,Amsterdam,Paris,8997,24.6,international
3027,Wien,Praha,8484,23.2,international
4207,Bruxelles,Luxembourg,6133,16.8,international
4340,Berlin,Praha,5963,16.3,international
6224,Berlin,Amsterdam,3708,10.2,international
6281,Berlin,Warszawa,3666,10.0,international
7150,Berlin,Wien,3014,8.3,international
